In [15]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
import subprocess
import time
import os

# Installation
print("Installing Ollama")
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!pip install -q ollama

# Starting the server
print("Starting Ollama Server (Background)")
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

# DOWNLOAD MODEL
print("Downloading Llama 3.1 (8B)")
!ollama pull llama3.1 > /dev/null
print("Model downloaded")

import ollama
print("Loading model into GPU memory")
ollama.chat(model='llama3.1', messages=[{'role':'user', 'content':'hi'}])

print("\nGPU STATUS")
!nvidia-smi | grep "ollama" || echo "Ollama is NOT visible in nvidia-smi (Check logs)"

print("\n OLLAMA PS (Process Status)")
!ollama ps

Installing Ollama
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama Server (Background)

Model downloaded
Loading model into GPU memory

GPU STATUS
Ollama is NOT visible in nvidia-smi (Check logs)

 OLLAMA PS (Process Status)
NAME               ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
llama3.1:latest    46e0c10c039e    5.5 GB    100% GPU     4096       4 minutes from now    


In [17]:
CSV_FILENAME = "/content/drive/MyDrive/basketball_transcript_corrected.csv"


In [18]:
import pandas as pd
import ollama
MODEL_NAME = "llama3.1"

print("Loading and formatting transcript...")

try:

    df = pd.read_csv(CSV_FILENAME)


    formatted_transcript = ""
    for index, row in df.iterrows():
        timestamp = row.iloc[0]
        text = row.iloc[2]

        formatted_transcript += f"[{timestamp}] {text}\n"

    print(f"Success! Transcript length: {len(formatted_transcript)} characters.")

except Exception as e:
    print(f"Error reading CSV: {e}")
    formatted_transcript = ""

Loading and formatting transcript...
Success! Transcript length: 97114 characters.


Tried to prompt with the whole transcript in a single prompt but the LLM hallucinates and doesn't give correct answers based on the transcript.
Wanted to try with a better LLM with more parameters but due to RAM and GPU restrictions, I just tried with just 20 minutes of the game's transcript. With just 20 minutes the answers were better and accurate.

In [19]:
#  OLLAMA
if formatted_transcript:
    print("\nSending to Llama 3.1")

    # Prompt
    query = """
    Analyze the player who scored the most points in this game.
    1. Identify the player.
    2. Describe their performance chronologically (Early vs Late game).
    3. EXTRACT QUOTES: Mention specific "narratives" or "underrated aspects" from the commentary.
    4. CITE TIMESTAMPS: You must use the timestamps provided, e.g., (12:30).
    """

    response = ollama.chat(
        model='llama3.1',
        messages=[
            {'role': 'system', 'content': "You are an expert sports analyst. Answer ONLY based on the transcript provided."},
            {'role': 'user', 'content': f"TRANSCRIPT:\n{formatted_transcript}\n\nREQUEST: {query}"}
        ],
        options={'num_ctx': 32768}
    )

    print("\n" + "="*40)
    print("Answer based on the prompt")
    print("="*40 + "\n")
    print(response['message']['content'])


Sending to Llama 3.1

Answer based on the prompt

**1. Identify the player who scored the most points in this game**

The player who scored the most points in this game is Steph Curry.

**2. Describe their performance chronologically (Early vs Late game)**

Steph Curry's performance can be broken down into two periods:

* Early Game: Curry struggled initially, taking only three shots and averaging 0 for 3 from the field. He didn't seem to get into a rhythm until later in the game.
* Late Game: In the second half, particularly in the fourth quarter, Curry erupted with a series of clutch shots, including five three-pointers. His performance was crucial in bringing his team back into contention.

**3. EXTRACT QUOTES**

Narrative 1:
"The most underrated two things about his [Steph Curry's] game are the rebounding aspect and the finishing aspect." (12:30)

Underrated Aspects:

* "The type of plays he makes, especially when they're defending him one-on-one" (14:04)
* "He gives you a great d




Sending to Llama 3.1

========================================
Answer based on the prompt
========================================

**1. Identify the player who scored the most points in this game**

The player who scored the most points in this game is Steph Curry.

**2. Describe their performance chronologically (Early vs Late game)**

Steph Curry's performance can be broken down into two periods:

* Early Game: Curry struggled initially, taking only three shots and averaging 0 for 3 from the field. He didn't seem to get into a rhythm until later in the game.
* Late Game: In the second half, particularly in the fourth quarter, Curry erupted with a series of clutch shots, including five three-pointers. His performance was crucial in bringing his team back into contention.

**3. EXTRACT QUOTES**

Narrative 1:
"The most underrated two things about his [Steph Curry's] game are the rebounding aspect and the finishing aspect." (12:30)

Underrated Aspects:

* "The type of plays he makes, especially when they're defending him one-on-one" (14:04)
* "He gives you a great defense, and he's got a great offense" (19:88)

Narrative 2:
"The way this team has been able to compete at a high level without Curry or Green is impressive." (27:52)

Underrated Aspects:

* "The ability of the Warriors to spread the floor with shooting all around" (30:00)
* "They have Toscano Anderson, who can really pass and make plays" (35:00)

**4. CITE TIMESTAMPS**

Timestamps are provided in the original transcript, but I will only include the relevant ones for this analysis.

Some notable timestamps include:

* 12:30 - Mention of Curry's early game struggles
* 14:04 - Discussion of Curry's underrated aspects
* 19:88 - Praise for Curry's all-around game
* 27:52 - Analysis of the Warriors' ability to compete without Curry or Green
* 35:00 - Description of Toscano Anderson's skills


In [23]:
import subprocess
import time
import os

# Installation
print("Installing Ollama")
!curl -fsSL https://ollama.com/install.sh | sh > /dev/null
!pip install -q ollama

# Starting the server
print("Starting Ollama Server (Background)")
os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
process = subprocess.Popen(["ollama", "serve"], stdout=subprocess.PIPE, stderr=subprocess.PIPE)
time.sleep(5)

# DOWNLOAD MODEL
print("Downloading Llama 3.1 (8B)")
!ollama pull llama3.1 > /dev/null
print("Model downloaded")

import ollama
print("Loading model into GPU memory")
ollama.chat(model='llama3.1', messages=[{'role':'user', 'content':'hi'}])

print("\nGPU STATUS")
!nvidia-smi | grep "ollama" || echo "Ollama is NOT visible in nvidia-smi (Check logs)"

print("\n OLLAMA PS (Process Status)")
!ollama ps

Installing Ollama
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
Starting Ollama Server (Background)

Model downloaded
Loading model into GPU memory

GPU STATUS
Ollama is NOT visible in nvidia-smi (Check logs)

 OLLAMA PS (Process Status)
NAME               ID              SIZE      PROCESSOR    CONTEXT    UNTIL              
llama3.1:latest    46e0c10c039e    5.5 GB    100% GPU     4096       4 minutes from now    


In [ ]:
import pandas as pd
import ollama
import re

MINUTES_LIMIT = 20

# LOAD
print(f" transcript (First {MINUTES_LIMIT} mins)... ", end="")
try:
    df = pd.read_csv(CSV_FILENAME)

    # Filter
    df['seconds'] = pd.to_numeric(df.iloc[:, 0], errors='coerce')
    limit_seconds = MINUTES_LIMIT * 60
    df_subset = df[df['seconds'] <= limit_seconds].copy()
    df_subset.reset_index(drop=True, inplace=True)

    searchable_text = ""
    for index, row in df_subset.iterrows():
        text_val = str(row.iloc[2]).strip()
        searchable_text += f"ID_{index}: {text_val}\n"

    print(f"Done! ({len(df_subset)})")

except Exception as e:
    print(f"\nError: {e}")
    searchable_text = ""
    df_subset = pd.DataFrame()

print("\n" + "="*40)
print(f" Enter the question")
print("="*40 + "\n")

while True:
    user_question = input("\nQuestion: ")
    if user_question.lower() in ['exit', 'quit']:
        break

    print("Please wait for the answer: ", end="")

    search_prompt = f"""
    TRANSCRIPT:
    {searchable_text}

    QUESTION: {user_question}

    TASK: Find the single best line ID that answers the question.
    OUTPUT: ONLY the ID (e.g. ID_100) or ID_NONE.
    """

    try:
        search_response = ollama.chat(
            model='llama3.1',
            messages=[{'role': 'user', 'content': search_prompt}],
            options={'num_ctx': 32768}
        )
        ai_output = search_response['message']['content'].strip()

        match = re.search(r'ID_(\d+)', ai_output, re.IGNORECASE)

        if match:
            row_id = int(match.group(1))

            if row_id < len(df_subset):
                raw_time = df_subset.iloc[row_id]['seconds']
                found_quote = df_subset.iloc[row_id, 2]

                # Format
                m, s = divmod(raw_time, 60)
                timestamp_str = f"{int(m):02d}:{int(s):02d}"

                final_prompt = f"""
                You are a sports commentator assistant.

                USER QUESTION: "{user_question}"
                FACT FOUND: The transcript says "{found_quote}" at timestamp [{timestamp_str}].

                INSTRUCTION: Write a natural, direct answer to the user's question using the Fact Found.
                - Always mention the timestamp explicitly.
                - Keep it short (1-2 sentences).
                """

                final_response = ollama.chat(
                    model='llama3.1',
                    messages=[{'role': 'user', 'content': final_prompt}]
                )

                print("\n")
                print(final_response['message']['content'])

            else:
                print("\nError: Outside the 20-minute limit.")

        elif "NONE" in ai_output:
            print("\n I couldn't find the answer")
        else:
            print("\n I'm not sure.")

    except Exception as e:
        print(f"\nError: {e}")

 transcript (First 20 mins)... Done! (490)

 Enter the question


Question: Who had the most number of points ?
Please wait for the answer: 

At timestamp [19:11], it was mentioned in the transcript that Michael Jordan had reached an incredible milestone by then. Unfortunately, it doesn't specifically say he held the record for most points scored, but we can infer his impressive performance is notable nonetheless.

Question: exit


 transcript (First 20 mins)... Done! (490)

========================================
 Enter the question
========================================


Question: When did Curry score his first goal ?
Please wait for the answer:

I think you might be mistaken. The transcript actually says "Steph Curry with his first" at timestamp [11:05], but there's no indication that he scored a goal in this instance. It's likely referring to one of his many achievements in basketball, not soccer.

Question: Who had the most number of points ?
Please wait for the answer: